# Decentralized Value Learning

Trains **N separate models** (one per agent) to predict agent-specific value functions V^(i)(s).

Supports:
- **TD(0)**: Bootstrap from target network
- **TD(λ)**: λ-returns with trajectory buffer

Uses the analytical closed-form solution for ground truth values.

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

# --- Environment ---
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
NUM_APPLES = 5

# --- Reward Scheme ---
REWARD_SCHEME = "minus1_2"  # "minus1_2", "picker_1", "other_1", "1_over_n"

# --- Value Learning ---
LEARNING_METHOD = "td0"  # "td0" or "td_lambda"
DISCOUNT = 0.99
LAMBDA = 0.9  # Only used if LEARNING_METHOD == "td_lambda"

# --- Model ---
MODEL_TYPE = "MLP"  # "CNN" or "MLP"
MLP_HIDDEN_DIM = 64
MLP_NUM_LAYERS = 2
CNN_CONV_CHANNELS = [8]
CNN_KERNEL_SIZE = 3
CNN_HEAD_HIDDEN_DIM = 32
CNN_HEAD_NUM_LAYERS = 1

# --- Training ---
LR = 0.0001
BATCH_SIZE = 64
MAX_STEPS = 200000
EVAL_FREQ = 2000
TARGET_UPDATE_FREQ = 1000
REPLAY_BUFFER_SIZE = 50000

# --- TD(λ) specific ---
TRAJECTORY_LENGTH = 100
NUM_TRAJECTORIES = 50

# --- Benchmarks ---
SOFT_THRESHOLD = 5.0
HARD_THRESHOLD = 10.0
NUM_TEST_SAMPLES = 500

# --- Output ---
EXPERIMENT_NAME = "value_decen"
OUTPUT_DIR = "outputs"
CSV_PATH = "outputs/experiment_results.csv"
SEED = 42

In [ ]:
import sys
import time
import ast
from pathlib import Path
from typing import List, Optional, Dict, Callable

import numpy as np
import torch
from tqdm import tqdm

sys.path.append("../../")

# Environment & Helpers
from tadd_helpers.env_functions import State, init_empty_state
from tadd_helpers.setting_seed import set_all_seeds
from teleport_dynamic.base_value_model import BaseValueModelV2
from teleport_dynamic.models_decen_5ch import ValueCNNDecentralized5Ch, ValueMLPDecentralized5Ch
from teleport_dynamic.rewards_decentralized import (
    get_reward_minus1_2, get_reward_picker_1, get_reward_other_1, get_reward_1_over_n
)
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent

# Training functions
from teleport_dynamic.training_functions import (
    train_td0,
    train_td_lambda,
    TrajectoryBuffer,
)

# Experiment utilities
from teleport_dynamic.experiment_utils import (
    generate_decentralized_value_test_set,
    evaluate_decentralized_models,
    check_benchmark_decentralized,
    append_experiment_result,
    format_decen_eval_results_for_log,
    get_current_ram_mb
)

if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

# Reward function mapping
REWARD_FUNCS: Dict[str, Callable[[State, int], Dict[int, float]]] = {
    "minus1_2": get_reward_minus1_2,
    "picker_1": get_reward_picker_1,
    "other_1": get_reward_other_1,
    "1_over_n": get_reward_1_over_n,
}
reward_func = REWARD_FUNCS[REWARD_SCHEME]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_all_seeds(SEED)

# Define Architecture String for Filename
if MODEL_TYPE == "CNN":
    chan_str = str(CNN_CONV_CHANNELS).replace(" ", "")
    arch_str = f"conv{chan_str}_k{CNN_KERNEL_SIZE}_head{CNN_HEAD_HIDDEN_DIM}x{CNN_HEAD_NUM_LAYERS}"
else:
    arch_str = f"mlp{MLP_HIDDEN_DIM}x{MLP_NUM_LAYERS}"

method_str = LEARNING_METHOD if LEARNING_METHOD == "td0" else f"tdl{LAMBDA}"
log_filename = f"{EXPERIMENT_NAME}_{method_str}_{MODEL_TYPE}_{WIDTH}x{HEIGHT}_{NUM_AGENTS}ag_{REWARD_SCHEME}_g{DISCOUNT}_{arch_str}.txt"

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUTPUT_DIR / log_filename

print(f"Device: {DEVICE}")
print(f"Learning Method: {LEARNING_METHOD}")
print(f"Discount (γ): {DISCOUNT}")
print(f"Reward Scheme: {REWARD_SCHEME}")
if LEARNING_METHOD == "td_lambda":
    print(f"Lambda (λ): {LAMBDA}")
print(f"Logging to: {LOG_FILE}")

In [ ]:
# Model Initialization - One per agent
models: List[BaseValueModelV2] = []

for i in range(NUM_AGENTS):
    if MODEL_TYPE == "CNN":
        m = ValueCNNDecentralized5Ch(
            height=HEIGHT, width=WIDTH, self_agent_idx=i,
            lr=LR, discount=DISCOUNT,
            mlp_hidden_dim=CNN_HEAD_HIDDEN_DIM, mlp_num_layers=CNN_HEAD_NUM_LAYERS,
            conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE,
            device=DEVICE, replay_buffer_capacity=REPLAY_BUFFER_SIZE
        )
        model_config = f"conv={CNN_CONV_CHANNELS},k={CNN_KERNEL_SIZE},head={CNN_HEAD_HIDDEN_DIM}"
    else:
        m = ValueMLPDecentralized5Ch(
            height=HEIGHT, width=WIDTH, self_agent_idx=i,
            lr=LR, discount=DISCOUNT,
            hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS,
            device=DEVICE, replay_buffer_capacity=REPLAY_BUFFER_SIZE
        )
        model_config = f"hidden={MLP_HIDDEN_DIM},layers={MLP_NUM_LAYERS}"
    models.append(m)

num_params = models[0].get_num_parameters()
print(f"Models: {NUM_AGENTS} x {num_params:,} params")
print(f"Config: {model_config}")

In [ ]:
# Initialize State and Test Set
state = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)

print("Generating value test set (computing analytical solutions)...")
test_sets = generate_decentralized_value_test_set(
    HEIGHT, WIDTH, NUM_AGENTS, NUM_APPLES, NUM_TEST_SAMPLES,
    state.apples, reward_func, DISCOUNT, SEED + 1000
)

# Show value statistics
all_values = []
for cases in test_sets.values():
    all_values.extend([c.true_value for c in cases])
all_values = np.array(all_values)

print(f"\nValue Statistics:")
print(f"  Min: {all_values.min():.4f}")
print(f"  Max: {all_values.max():.4f}")
print(f"  Mean: {all_values.mean():.4f}")
print(f"  Std: {all_values.std():.4f}")
print(f"\nTest set generated: {sum(len(v) for v in test_sets.values())} samples")

In [ ]:
# Initialize trajectory buffers for TD(λ) - one per agent
trajectory_buffers: Optional[List[TrajectoryBuffer]] = None
if LEARNING_METHOD == "td_lambda":
    trajectory_buffers = [
        TrajectoryBuffer(
            max_trajectories=NUM_TRAJECTORIES,
            max_trajectory_length=TRAJECTORY_LENGTH
        )
        for _ in range(NUM_AGENTS)
    ]
    print(f"Trajectory buffers initialized: {NUM_AGENTS} x {NUM_TRAJECTORIES} x {TRAJECTORY_LENGTH}")

In [ ]:
# Training Loop
with open(LOG_FILE, "w") as f:
    f.write(f"START Decentralized Value Learning: {EXPERIMENT_NAME}\n")
    f.write(f"Method: {LEARNING_METHOD} | Scheme: {REWARD_SCHEME}\n")
    f.write(f"Grid: {WIDTH}x{HEIGHT}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}\n")
    f.write(f"Discount: {DISCOUNT}, LR: {LR}\n")
    if LEARNING_METHOD == "td_lambda":
        f.write(f"Lambda: {LAMBDA}\n")
    f.write(f"Params per model: {num_params:,} | Config: {model_config}\n")
    f.write(f"Benchmarks: Soft < {SOFT_THRESHOLD}%, Hard < {HARD_THRESHOLD}%\n")
    f.write("="*80 + "\n")

soft_benchmark_step = None
hard_benchmark_step = None
last_avg_loss = 0.0
start_time = time.time()
nan_detected = False

# Initialize c_t randomly
c_t = np.random.randint(0, NUM_AGENTS)

print(f"Starting {LEARNING_METHOD.upper()} training for {MAX_STEPS} steps...")

for step in range(MAX_STEPS):
    # Get rewards for all agents
    rewards = reward_func(state, c_t)
    
    # Transition
    next_state = state.copy()
    teleport_agent(next_state, c_t)
    c_t_next = np.random.randint(0, NUM_AGENTS)
    
    # Train each agent's model
    step_losses = []
    for i in range(NUM_AGENTS):
        # Add experience to replay buffer
        models[i].add_experience(state, next_state, rewards[i], c_t, c_t_next)
        
        # Also add to trajectory buffer for TD(λ)
        if LEARNING_METHOD == "td_lambda" and trajectory_buffers is not None:
            s_enc = models[i].raw_state_to_nn_input(state, c_t)
            ns_enc = models[i].raw_state_to_nn_input(next_state, c_t_next)
            trajectory_buffers[i].add_step(s_enc, ns_enc, rewards[i])
        
        # Train
        loss = None
        if LEARNING_METHOD == "td0":
            loss = train_td0(models[i], BATCH_SIZE)
        elif LEARNING_METHOD == "td_lambda" and trajectory_buffers is not None:
            if len(trajectory_buffers[i]) >= 5:
                loss = train_td_lambda(
                    models[i], trajectory_buffers[i], DISCOUNT, LAMBDA, BATCH_SIZE
                )
        
        if loss is not None:
            step_losses.append(loss)
            if np.isnan(loss) or np.isinf(loss):
                nan_detected = True
    
    if nan_detected:
        print(f"\nNaN/Inf detected at step {step}! Stopping early.")
        with open(LOG_FILE, "a") as f:
            f.write(f"\nNaN/Inf detected at step {step}! Training stopped.\n")
        break
    
    if step_losses:
        last_avg_loss = float(np.mean(step_losses))
    
    # Update target networks
    if step > 0 and step % TARGET_UPDATE_FREQ == 0:
        for m in models:
            m.update_target_net()
    
    # Advance state
    state = next_state
    c_t = c_t_next
    
    # Evaluation
    if step > 0 and step % EVAL_FREQ == 0:
        results = evaluate_decentralized_models(models, test_sets, use_value=True)
        soft, hard = check_benchmark_decentralized(results, SOFT_THRESHOLD, HARD_THRESHOLD)
        ram_mb = get_current_ram_mb()
        
        if soft and soft_benchmark_step is None:
            soft_benchmark_step = step
            print(f"Soft benchmark achieved at step {step}!")
        if hard and hard_benchmark_step is None:
            hard_benchmark_step = step
            print(f"Hard benchmark achieved at step {step}!")
        
        bench_str = "HARD" if hard_benchmark_step else ("Soft" if soft_benchmark_step else "-")
        
        log_line = format_decen_eval_results_for_log(results, step, last_avg_loss, NUM_AGENTS)
        log_line += f" | RAM: {ram_mb:.1f}MB | Bench: {bench_str}"
        
        with open(LOG_FILE, "a") as f:
            f.write(log_line + "\n")
        
        if hard_benchmark_step is not None:
            break

# Final evaluation
wall_time = time.time() - start_time
final_results = evaluate_decentralized_models(models, test_sets, use_value=True)

all_means = []
all_maxes = []
for cat_results in final_results.values():
    for agent_result in cat_results.values():
        all_means.append(agent_result.mean_pct_error)
        all_maxes.append(agent_result.max_pct_error)

final_mean = max(all_means) if all_means else float('nan')
final_max = max(all_maxes) if all_maxes else float('nan')

# Log to CSV
method_note = f"{LEARNING_METHOD}_{REWARD_SCHEME}"
if LEARNING_METHOD == "td_lambda":
    method_note += f"_l{LAMBDA}"
if nan_detected:
    method_note += "_NaN"

append_experiment_result(
    CSV_PATH, f"value_{LEARNING_METHOD}", MODEL_TYPE, False,
    f"{WIDTH}x{HEIGHT}", NUM_AGENTS, NUM_APPLES, f"{REWARD_SCHEME}_g{DISCOUNT}",
    model_config, soft_benchmark_step, hard_benchmark_step,
    step, final_mean, final_max, wall_time, method_note
)

print(f"\nTraining complete!")
print(f"Final Mean Error: {final_mean:.2f}%")
print(f"Final Max Error: {final_max:.2f}%")
print(f"Wall Time: {wall_time:.1f}s")
print(f"Log saved to: {LOG_FILE}")

In [ ]:
# Sanity Check: Visualize predictions vs analytical values
print("\n" + "="*80)
print("SANITY CHECK: Sample Predictions vs Analytical Values")
print("="*80)

for category, cases in test_sets.items():
    if not cases:
        continue
    
    print(f"\n>>> CATEGORY: {category.upper()} <<<")
    samples = cases[:3]
    
    for i, case in enumerate(samples):
        agent_model = models[case.self_agent_idx]
        pred = agent_model.get_value(case.state, case.acting_agent_idx)
        actual = case.true_value
        error = abs(pred - actual)
        pct_err = (error / abs(actual)) * 100 if abs(actual) > 1e-9 else error * 100
        
        print(f"\n--- Sample {i+1} ---")
        print(f"Self Agent: {case.self_agent_idx}, Acting Agent: {case.acting_agent_idx}")
        print(f"Immediate Reward: {case.true_reward:.4f}")
        print(f"PREDICTED V^({case.self_agent_idx})(s): {pred:.4f}")
        print(f"ANALYTICAL V^({case.self_agent_idx})(s): {actual:.4f}")
        print(f"ERROR: {error:.4f} ({pct_err:.2f}%)")

In [ ]:
# Plot loss history for each agent
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, NUM_AGENTS, figsize=(5*NUM_AGENTS, 4))
if NUM_AGENTS == 1:
    axes = [axes]

for i, (ax, model) in enumerate(zip(axes, models)):
    if len(model.loss_history) > 0:
        history = model.loss_history
        if len(history) > 5000:
            step_size = len(history) // 5000
            history = history[::step_size]
        ax.plot(history)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Loss')
        ax.set_title(f'Agent {i} Loss')
        ax.set_yscale('log')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(f'Agent {i}')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{log_filename.replace('.txt', '_loss.png')}")
plt.show()